# Nearest-TSS Gene Evaluation — 26.03 Training Set

Assesses what fraction of the gold-standard positives in the final training
set (`2603_training_set_full_fm.parquet`) are the **closest gene to the
sentinel variant TSS** in their credible set.

The feature `distanceSentinelTssNeighbourhood` equals **1** when the gene
has the smallest TSS distance to the sentinel variant among all genes in
the credible-set neighbourhood, and **0** otherwise.

Two levels of evaluation:
1. **Row level** — every positive (studyLocusId, geneId) row
2. **Unique gene–disease level** — each distinct (geneId, diseaseId) pair,
   where a pair counts as "nearest TSS" if it is nearest in **any** of its
   credible sets

In [ ]:
import pyspark.sql.functions as f

In [ ]:
"""Common utilities for the project."""

import os
from pathlib import Path
from gentropy.common.session import Session
import logging


def get_gcs_credentials() -> str:
    app_default_credentials = os.path.join(
        os.getenv("HOME", "."), ".config/gcloud/application_default_credentials.json"
    )
    if Path(app_default_credentials).exists():
        return app_default_credentials
    raise FileNotFoundError("No GCS credentials found.")


def get_gcs_hadoop_connector_jar() -> str:
    return (
        "https://storage.googleapis.com/hadoop-lib/gcs/gcs-connector-hadoop3-latest.jar"
    )


def gcs_conf(
    credentials_path=None, project="open-targets-genetics-dev"
) -> dict[str, str]:
    credentials_path = credentials_path or get_gcs_credentials()
    return {
        "spark.driver.memory": "12g",
        "spark.kryoserializer.buffer.max": "500m",
        "spark.driver.maxResultSize": "2g",
        "spark.hadoop.fs.gs.impl": "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystem",
        "spark.jars": get_gcs_hadoop_connector_jar(),
        "spark.hadoop.google.cloud.auth.service.account.enable": "true",
        "spark.hadoop.fs.gs.project.id": project,
        "spark.hadoop.google.cloud.auth.service.account.json.keyfile": credentials_path,
        "spark.hadoop.fs.gs.requester.pays.mode": "AUTO",
    }


class GentropySession(Session):
    def __init__(self, *args, **kwargs):
        if "extended_spark_conf" in kwargs:
            kwargs["extended_spark_conf"].update(gcs_conf())
        else:
            kwargs["extended_spark_conf"] = gcs_conf()
        super().__init__(*args, **kwargs)


session = GentropySession()

In [ ]:
fm = session.spark.read.parquet("2603_training_set_full_fm.parquet").cache()
positives = fm.filter(f.col("GSP") == 1).cache()
print(f"Total training set rows:  {fm.count():>10,}")
print(f"Total positive rows:      {positives.count():>10,}")

## 1. Row-Level Evaluation

Each row is one (studyLocusId, geneId) pair. The question is: among all
positive rows, how many have the gene as the nearest TSS gene in that
credible set?

In [ ]:
n_pos = positives.count()
n_nearest = positives.filter(f.col("distanceSentinelTssNeighbourhood") == 1).count()
n_not_nearest = n_pos - n_nearest

print(f"Positive rows total:          {n_pos:>8,}")
print(f"  nearest TSS  (feature = 1): {n_nearest:>8,}  ({n_nearest / n_pos * 100:.1f}%)")
print(f"  not nearest  (feature = 0): {n_not_nearest:>8,}  ({n_not_nearest / n_pos * 100:.1f}%)")

## 2. Unique Gene–Disease Level

Collapse to distinct (geneId, diseaseId) pairs by exploding the `diseaseIds`
array. A pair is labelled **nearest TSS** if `distanceSentinelTssNeighbourhood == 1`
in **at least one** of its credible sets (max aggregation).

In [ ]:
# Explode diseaseIds array → one row per (geneId, diseaseId, studyLocusId)
pos_exploded = positives.withColumn("diseaseId", f.explode(f.col("diseaseIds")))

# For each unique (geneId, diseaseId) pair take the max nearest-TSS flag
# across all credible sets — 1 if nearest in any CS, 0 if never nearest
pair_nearest = (
    pos_exploded
    .groupBy("geneId", "diseaseId")
    .agg(f.max("distanceSentinelTssNeighbourhood").alias("nearest_tss"))
    .cache()
)

n_pairs        = pair_nearest.count()
n_pairs_near   = pair_nearest.filter(f.col("nearest_tss") == 1).count()
n_pairs_far    = n_pairs - n_pairs_near

print(f"Unique gene-disease pairs:    {n_pairs:>8,}")
print(f"  nearest TSS in any CS:      {n_pairs_near:>8,}  ({n_pairs_near / n_pairs * 100:.1f}%)")
print(f"  never nearest TSS:          {n_pairs_far:>8,}  ({n_pairs_far / n_pairs * 100:.1f}%)")

## 3. Summary Table

In [ ]:
from pyspark.sql import Row

rows = [
    Row(level="Row level (studyLocusId × geneId)",
        total=n_pos,        nearest=n_nearest,    pct_nearest=round(n_nearest/n_pos*100,1)),
    Row(level="Unique gene-disease pairs",
        total=n_pairs,      nearest=n_pairs_near, pct_nearest=round(n_pairs_near/n_pairs*100,1)),
]

summary = session.spark.createDataFrame(rows)
summary.show(truncate=False)